## Authentication

`docker login <registry>` prompts for a username and password (or token), then stores the credential so later `push`/`pull` to that registry authenticate automatically:

```bash
docker login                    # defaults to docker.io
docker login ghcr.io            # a specific registry
echo "$TOKEN" | docker login -u USER --password-stdin ghcr.io   # non-interactive, safe
docker logout ghcr.io           # forget it
```

Prefer **`--password-stdin`** over `-p TOKEN` on the command line — the latter lands your token in shell history and the process list.

**Where credentials go — and the catch.** By default they're written to `~/.docker/config.json`:

```json
{ "auths": { "ghcr.io": { "auth": "base64(user:token)" } } }
```

That `auth` value is **just base64, not encryption** — anyone who reads the file has your credentials. Base64 is encoding, not secrecy.

**The right answer: a credential helper**, which offloads storage to the OS keychain or a secret manager instead of a plaintext file. `config.json` then holds a `credsStore` name, not the secret:

- **macOS** — `docker-credential-osxkeychain` (Docker Desktop installs it)
- **Windows** — `docker-credential-wincred`
- **Linux** — `docker-credential-pass` or `-secretservice`
- **Cloud** — `docker-credential-ecr-login` (uses AWS IAM), and equivalents for GAR/ACR that mint short-lived tokens from cloud identity

In CI, you don't type `docker login` at all — the platform injects a short-lived token (a GitHub Actions `GITHUB_TOKEN`, an ECR login via IAM role) so no long-lived password sits in the pipeline. The principle across all of it: **keep registry credentials out of plaintext and short-lived where you can** — a theme module 08 returns to for secrets in general.